# GPT-Neo residual stream — every block, in memory

A decoder-only model, capturing every block at once: `n_layers x seq_len x
hidden` floats per passage. Token-shaped activations flatten padding away, so
the accumulated tensors only ever hold real tokens.

Downloads on first run: WikiText-2 (~5 MB) and GPT-Neo 125M weights (~500 MB).

In [ ]:
import sys
from pathlib import Path

# Make the repo-local examples._utils package importable when this notebook
# is opened directly from examples/text/, without installing anything extra.
sys.path.insert(0, str(Path.cwd().parents[1]))

from transformers import AutoModelForCausalLM, AutoTokenizer

from examples._utils.data import activation_loader
from examples._utils.text import WikiTextSamples
from nnact import ActivationPipeline
from nnact._model._hooked import HookedModel

MODEL = "EleutherAI/gpt-neo-125m"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token  # GPT-Neo ships without a pad token
model = AutoModelForCausalLM.from_pretrained(MODEL)  # has .logits, real next-token head


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125m
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
hooked = HookedModel(model)

# The residual stream: every transformer block, plus the final layer norm.
num_layers = model.config.num_layers
LAYERS = [f"transformer.h.{i}" for i in range(num_layers)] + ["transformer.ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")

512 passages | 13 layers: ['transformer.h.0', 'transformer.h.1', 'transformer.h.2', 'transformer.h.3', 'transformer.h.4', 'transformer.h.5', 'transformer.h.6', 'transformer.h.7', 'transformer.h.8', 'transformer.h.9', 'transformer.h.10', 'transformer.h.11', 'transformer.ln_f']


In [ ]:
# run() accumulates every batch's activations in memory as the run
# progresses, flattened to one row per real token (padding dropped), and
# hands back the finished ActivationDataset. run_dir is required for every
# run: it holds run.log and run_metadata.json.
pipeline = ActivationPipeline(
    model,
    LAYERS,
    output_type="token",
    run_dir=Path.cwd() / "runs" / "02_gpt2_residual_stream",
)
loader = activation_loader(dataset, batch_size=16)
activations = pipeline.run(loader)
activations.summary()


In [ ]:
# Slicing by offsets pulls out one passage's own tokens across every layer.
import numpy as np

offsets = activations.offsets
start, end = int(offsets[0]), int(offsets[1])
sample_activations = {
    name: tensor[start:end] for name, tensor in activations.activations.items()
}
print(
    "first passage ->",
    len(sample_activations),
    "layers,",
    tuple(next(iter(sample_activations.values())).shape),
    "each",
)

# Residual stream norm grows with depth - the usual GPT-2/GPT-Neo picture.
for name, tensor in sample_activations.items():
    print(f"  {name:<18} mean L2 norm = {np.linalg.norm(tensor, axis=-1).mean():6.2f}")